# PawCity AI — Complete Jupyter Workflow

**IEOR 4570 · Project 2 · Deep Research path**

**Research question:** Which NYC ZIP-code areas have limited documented dog-run access relative to recorded dog-license periods and incident reports, and which conclusions are unsupported?

**Who this is for:** NYC dog owners, urban planners, and public-data researchers.

This notebook verifies **four genuine, included NYC Open Data extracts**, tests the evidence tools, and provides optional **real LLM research**. The final website is served by `app.py`; the Cloud Run container uses the same Python source files. No individual owner-level data or imaginary research results are used.

## 1. Locate the project

Run the notebook from the directory containing `app.py`, `pawcity/` and `data/`. No individual file copying is required.

In [1]:
from pathlib import Path
import os, sys, json, csv, subprocess

ROOT = Path.cwd()
if not (ROOT / "app.py").exists() and (ROOT / "PawCity_AI_Complete" / "app.py").exists():
    ROOT = ROOT / "PawCity_AI_Complete"
if not (ROOT / "app.py").exists():
    raise FileNotFoundError("Open this notebook in the PawCity_AI_Complete project directory.")
sys.path.insert(0, str(ROOT))
print("Project directory:", ROOT.resolve())
print("Python version:", sys.version.split()[0])

Project directory: /mnt/data/PawCity_AI_Complete
Python version: 3.13.5


## 2. Inspect the real data provenance

We record where each extract came from and what it actually measures. The 311 extract is **2025 Barking Dog noise requests only**, not all dog-related service requests.

In [2]:
log = json.loads((ROOT / "data/metadata/query_log.json").read_text(encoding="utf-8"))
for source in ("dog_runs", "dog_licenses", "dog_311", "dog_bites"):
    meta = log[source]
    print(f"{source:14} status={meta['status']:5} grouped/facility rows={meta['rows']:,} | dataset={meta['dataset_id']}")
assert all(log[name]["status"] == "ok" for name in ("dog_runs", "dog_licenses", "dog_311", "dog_bites"))
assert log["dog_311"]["event_year"] == 2025
assert "Barking Dog" in log["dog_311"]["complaint_scope"]
print("Data provenance verification: PASS")

dog_runs       status=ok    grouped/facility rows=95 | dataset=hxx3-bwgv
dog_licenses   status=ok    grouped/facility rows=19,091 | dataset=nu7n-tubp
dog_311        status=ok    grouped/facility rows=180 | dataset=erm2-nwe9
dog_bites      status=ok    grouped/facility rows=241 | dataset=rsgh-akpg
Data provenance verification: PASS


## 3. Check aggregate totals and quality notes

Counts are descriptive. License periods do not represent independent dogs. ZIP strings may include owner addresses beyond NYC and do not define neighborhood boundaries.

In [3]:
from pawcity.pipeline import read_csv, make_summary_report
from pawcity.research_agent import DataTools

make_summary_report(ROOT)
tools = DataTools(ROOT)
report = tools.inventory()
print("Distinct dog-run facility records:", report["distinct_dog_run_records"])
print("2025 Barking Dog requests:", report["barking_2025_requests"])
print("ZIP strings across source extracts:", report["all_zip_strings_across_sources"])
print("Conservatively scoped NYC ZIP strings:", report["nyc_zip_rows_conservative"])
print("\nInterpretation limits:\n", report["limitations"])
assert report["distinct_dog_run_records"] == log["dog_runs"]["rows"]
assert report["barking_2025_requests"] == sum(log["dog_311"]["monthly_counts"].values())

Distinct dog-run facility records: 95
2025 Barking Dog requests: 8327
ZIP strings across source extracts: 428
Conservatively scoped NYC ZIP strings: 228

Interpretation limits:
 ZIP areas are not neighborhoods or walking-distance measures. Some dog runs are tagged with multiple ZIPs; summing ZIP counts double-counts such facilities. License periods are not unique dogs and use the 2026 extract; barking requests and bite incidents are from 2025. Barking data covers ONLY 'Noise, Barking Dog (NR5)' requests; it is not all dog-related 311 activity. Complaint and bite counts are subject to reporting biases. License ZIP indicates the owner's home while incident ZIP indicates the event location. These records cannot establish dog safety, causation, per-dog risk, or an overall neighborhood ranking.


## 4. Verify tool-grounded comparisons

Every value below comes from a shipped CSV file. The tools return source IDs, file paths and metadata. These are not LLM predictions.

In [4]:
selected_zips = ["10001", "10002"]
comparison = {"ZIP": selected_zips}
for domain in ("parks", "licenses", "barking", "bites"):
    evidence = tools.query(domain, selected_zips)
    comparison[domain] = [x["value"] for x in evidence["rows"]]
    print(domain, "source:", evidence["source_id"], "values:", comparison[domain])
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(comparison))
except ImportError:
    print(comparison)
print("\nThese numbers are not comparable rates and do not establish safety.")

parks source: PARKS values: [0, 2]
licenses source: LICENSES_2026 values: [811, 1070]
barking source: BARKING_2025 values: [45, 72]
bites source: BITES_2025 values: [15, 16]


,ZIP,parks,licenses,barking,bites
0,10001,0,811,45,15
1,10002,2,1070,72,16



These numbers are not comparable rates and do not establish safety.


## 5. Test the research workflow without using an API key

These unit tests use explicitly labeled **fake model outputs** to validate the agent orchestration, including fresh-context parallel Researchers, Lead gap review, clarification, citations and failures when sources are missing. Passing mock tests does not prove the live model's factual accuracy.

In [5]:
check = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                       cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
print(check.stderr)
assert check.returncode == 0, "Offline tests failed"
print("All offline tests passed. Live model verification is a separate step.")

  Month 01: 2 requests
  Month 02: 2 requests
  Month 03: 2 requests
  Month 04: 2 requests
  Month 05: 2 requests
  Month 06: 2 requests
  Month 07: 2 requests
  Month 08: 2 requests
  Month 09: 2 requests
  Month 10: 2 requests
  Month 11: 2 requests
  Month 12: 2 requests

test_311_monthly_merge (test_download_data.TestDownload.test_311_monthly_merge) ... ok
test_chat_reports_missing_key (test_http_api.TestHTTPAPI.test_chat_reports_missing_key) ... 127.0.0.1 - - [11/Nov/2026 00:16:28] "POST /chat HTTP/1.1" 503 -
ok
test_chat_response_has_agent_tool_calls (test_http_api.TestHTTPAPI.test_chat_response_has_agent_tool_calls) ... 127.0.0.1 - - [11/Nov/2026 00:16:28] "POST /chat HTTP/1.1" 200 -
ok
test_health_data_available (test_http_api.TestHTTPAPI.test_health_data_available) ... 127.0.0.1 - - [11/Nov/2026 00:16:28] "GET /health HTTP/1.1" 200 -
ok
test_aggregated_sources_and_multi_zip (test_pipeline.TestSummary.test_aggregated_sources_and_multi_zip) ... ok
test_unavailable_source_not_ze

## 6. Inspect the citywide selection method

The agent never silently converts neighborhood names into ZIP codes. For a citywide question it produces a **labeled exploratory shortlist** (not a safety ranking) and queries each specialist on the same ZIPs.

In [6]:
questions = [
    "Which NYC ZIP codes have the most documented dog runs?",
    "Which NYC ZIP areas have no dog runs despite many license periods?",
    "Which ZIP codes have the highest 2025 barking-noise requests?",
]
for question in questions:
    zip_codes, reason = tools.candidates(question)
    print("\nQuestion:", question)
    print("Selection method:", reason)
    print("ZIPs:", ", ".join(zip_codes))
    print("Parks evidence:", [(x["zip_code"], x["value"]) for x in tools.query("parks", zip_codes)["rows"]])


Question: Which NYC ZIP codes have the most documented dog runs?
Selection method: Highest tagged dog-run count among conservatively scoped NYC ZIPs
ZIPs: 10024, 10040, 10471, 11201, 10002, 10003, 10011, 10019
Parks evidence: [('10024', 5), ('10040', 3), ('10471', 3), ('11201', 3), ('10002', 2), ('10003', 2), ('10011', 2), ('10019', 2)]

Question: Which NYC ZIP areas have no dog runs despite many license periods?
Selection method: Top ZIPs with zero tagged dog runs sorted by license periods (not dog counts)
ZIPs: 10025, 10023, 10128, 11238, 10314, 11217, 10021, 10306
Parks evidence: [('10025', 0), ('10023', 0), ('10128', 0), ('11238', 0), ('10314', 0), ('11217', 0), ('10021', 0), ('10306', 0)]

Question: Which ZIP codes have the highest 2025 barking-noise requests?
Selection method: Highest recorded 2025 barking requests among conservatively scoped NYC ZIPs
ZIPs: 11206, 10461, 10306, 11221, 11201, 11238, 11225, 11226
Parks evidence: [('11206', 1), ('10461', 0), ('10306', 0), ('11221',

## 7. Optional: call the real Lead, Researchers and Writer

This step uses paid model API calls. Your ChatGPT subscription does not automatically provide API credit. **Do not store an API key in this notebook.** Start JupyterLab from a terminal/session where `OPENAI_API_KEY` is already set as an environment variable, or configure the key securely through your hosting environment.

If the key is missing, this cell reports that the live model has been skipped; it does not show made-up research results.

In [7]:
from pawcity.research_agent import PawCityResearch

if os.getenv("OPENAI_API_KEY"):
    live_agent = PawCityResearch(ROOT, max_turns=2)
    live_result = live_agent.run("Compare ZIP 10001 and ZIP 10002 for dog runs, 2025 barking reports, license periods and bite reports.")
    print("REAL LLM RESEARCH ANSWER:\n", live_result.answer)
    print("\nSource citations:", json.dumps(live_result.citations, indent=2))
    print("Tool call agents:", [x["agent"] for x in live_result.tool_calls])
else:
    print("SKIPPED: OPENAI_API_KEY is not configured. Offline source verification remains valid.")

SKIPPED: OPENAI_API_KEY is not configured. Offline source verification remains valid.


## 8. Optional: run resumable, same-model comparisons

The three required comparisons use the **same model** for a bare-question baseline (no tools, no system prompt) and the real tool-assisted agent.

The process uses a conservative API request pace (`PAWCITY_API_RPM=7` by default) and writes a checkpoint after every baseline and completed agent response. If interrupted by a rate limit, **rerun the same cell**: saved responses are reused and completed questions are skipped.

Running this cell makes real, billable API calls. The script updates the README table only when all three comparisons are complete.


In [8]:
if os.getenv("OPENAI_API_KEY"):
    print("To run the three resumable comparisons in a new notebook cell:")
    print('import runpy; runpy.run_path(str(ROOT / "scripts" / "compare_samples.py"), run_name="__main__")')
    print("Re-run that cell if interrupted; saved responses will be reused.")
else:
    print("SKIPPED: API credentials required for actual model comparison.")


SKIPPED: API credentials required to generate actual model-vs-agent comparisons.


## 9. Open the custom web frontend and prepare deployment

**Terminal command:** `python app.py`  
**Local URL:** `http://localhost:8080`  
**Health check:** `http://localhost:8080/health`  
**Research endpoint:** `POST /chat` with `{"message": "your question"}`.

The frontend displays source status, aggregate data, suggested prompts, cited answers and all sub-agent tool calls. Without `OPENAI_API_KEY`, the read-only explorer works but the live `/chat` route intentionally reports that the model is not configured.

**Before submission:** obtain the Project 1 starter and verify the exact `/chat` response contract; run the real model comparisons; deploy via GitHub continuous deployment to Google Cloud Run; supply the actual Cloud Run URL and all team UNIs in `submission.json`; and verify the public URL until grades are released.

Read `README.md`, `DATA_DICTIONARY.md`, and `RUN_IN_JUPYTER.md` for submission instructions.